# 🏥 Insurance RAG — Centralized Pipeline Engine
**HDFC Ergo Optima Secure vs Care Insurance Supreme**

This notebook provides a complete end-to-end sandbox. It handles:
1. Pulling the backend code from GitHub.
2. Downloading the raw policy PDFs.
3. Extracting, chunking, and loading the documents into ChromaDB.
4. Querying the policies using our centralized `InsuranceRAGEngine`.

By using the central engine, this notebook behaves identically to the FastAPI backend and DeepEval test suites.

## 1 · Environment Setup & Repository Clone

In [ ]:
import os

# ── Pull GitHub Repository ──────────────────────────────────────────────────
# Only clone if we aren't already inside the repository directory
if not os.path.exists('engine.py'):
    print("Cloning repository...")
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
        !git clone https://{token}@github.com/falcon978/Insurance-RAG.git --quiet
    except Exception:
        # Fallback to public clone if token isn't provided
        !git clone https://github.com/falcon978/Insurance-RAG.git --quiet
    
    # Change working directory into the repo
    %cd Insurance-RAG
    print("✅ Cloned and entered Insurance-RAG repository.")
else:
    print("✅ Repository already present.")

In [ ]:
# ── Install Dependencies ────────────────────────────────────────────────────
!pip install -r requirements.txt --quiet

# nest_asyncio is required for FastAPI/async routines to run inside Jupyter's event loop
import nest_asyncio
nest_asyncio.apply()

import sys

# 1. Strip the corrupted versions from the active executable
!{sys.executable} -m pip uninstall -y numpy scipy spacy scikit-learn pandas

# 2. Install the mathematically locked versions directly into the kernel
!{sys.executable} -m pip install numpy==1.26.4 scipy==1.12.0 "spacy<3.8.0" "scikit-learn<1.5.0" "pandas<2.2.0" --quiet

print('✅ Dependencies installed and async patched.')

In [ ]:
import logging
import sys

logging.basicConfig(level=logging.INFO, stream=sys.stdout)
logging.getLogger("sentence_transformers").setLevel(logging.WARNING)

In [ ]:
import os
import sys

# Ensure the notebook can import your local modules
sys.path.append(os.path.abspath('.'))

from google.colab import userdata
try:
    os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
except Exception:
    os.environ["GEMINI_API_KEY"] = input("Enter your Google Gemini API Key: ")

# 2. Configure Pinecone
os.environ["VECTOR_DB_TYPE"] = "pinecone" # <--- This tells config.py to switch
os.environ["PINECONE_INDEX_NAME"] = "health-insurance-policies" # Replace with your actual Pinecone index name
os.environ["HF_DEVICE"] = "cuda"

try:
    os.environ["PINECONE_API_KEY"] = userdata.get('PINECONE_API_KEY')
except Exception:
    os.environ["PINECONE_API_KEY"] = input("Enter your Pinecone API Key: ")

# 3. Configure Lexical DB (Upstash RediSearch)
os.environ["LEXICAL_DB_TYPE"] = "upstash"

try:
    os.environ["UPSTASH_REDIS_URL"] = userdata.get('UPSTASH_REDIS_URL')
except Exception:
    os.environ["UPSTASH_REDIS_URL"] = input("Enter your Upstash Redis URL (rediss://...): ")

# 4. Configure LangSmith Tracing
os.environ["LANGSMITH_TRACING_V2"] = "true"
os.environ["LANGSMITH_PROJECT"] = "Insurance-RAG" # Change this to whatever project name you prefer

try:
    os.environ["LANGSMITH_API_KEY"] = userdata.get('LANGSMITH_API_KEY')
except Exception:
    os.environ["LANGSMITH_API_KEY"] = input("Enter your LangSmith API Key: ")

print("✅ Environment variables configured.")

In [ ]:
%cd /content/Insurance-RAG
!git pull origin main

In [ ]:
import importlib

# 1. Import the base modules themselves (required for importlib to target them)
import rag.generator
import rag_ingestion.pipeline

# 2. Force Python to flush the cache and reload the files from the hard drive
importlib.reload(rag.generator)
importlib.reload(rag_ingestion.pipeline)

# 3. Re-import the specific classes you are actually using in the notebook
from rag.generator import ResponseGenerator
from rag_ingestion.pipeline import ExtractionPipeline

print("✅ Modules manually reloaded! Notebook is now using the latest code.")

## 2 · Download Policy Documents
Fetch the raw PDFs if they aren't already downloaded.

In [ ]:
import httpx
from pathlib import Path

PDFS = {
    "optima_secure": {
        "name"  : "HDFC Ergo Optima Secure",
        "url"   : (
            "https://customer-portal-assets.hdfcergo.com/assets/v2/docs/"
            "default-source/downloads/policy-wordings/health/"
            "optima-secure-revision/optima-secure-revision-pw-647504209314.pdf"
        ),
        "path"  : "hdfc_optima_secure.pdf",
    },
    "care_supreme": {
        "name"  : "Care Insurance Supreme",
        "url"   : (
            "https://cms.careinsurance.com/cms/public/uploads/download_center/"
            "care-supreme---policy-terms-&-conditions-(effective-from-19-march-2025).pdf"
            "?rv=0.86869200%201775054695"
        ),
        "path"  : "care_supreme.pdf",
    },
}

async def download_pdfs():
    # Use AsyncClient to match the non-blocking behavior of the FastAPI backend
    async with httpx.AsyncClient() as client:
        for key, info in PDFS.items():
            if not Path(info['path']).exists():
                print(f"Downloading {info['name']} …")
                try:
                    response = await client.get(
                        info['url'],
                        headers={'User-Agent': 'Mozilla/5.0'},
                        follow_redirects=True,  # Crucial for robust URL resolution
                        timeout=60.0
                    )
                    response.raise_for_status()
                    Path(info['path']).write_bytes(response.content)
                    
                    size = Path(info['path']).stat().st_size // 1024
                    print(f"  ✅ {info['path']}  ({size} KB)")
                except httpx.HTTPError as exc:
                    print(f"  ❌ Network/HTTP Error for {info['name']}: {exc}")
                except Exception as e:
                    print(f"  ❌ Failed to write {info['name']}: {e}")
            else:
                print(f"✅ {info['path']} already exists")

# Execute the async function (Jupyter allows top-level await)
await download_pdfs()

## 3 · Data Ingestion: Extract, Chunk & Index
Process the downloaded PDFs into semantic chunks and store them in ChromaDB using our `ExtractionPipeline`.

In [ ]:
from rag_ingestion.pipeline import ExtractionPipeline

CHUNK_SIZE = 2200
OVERLAP    = 400

for key, info in PDFS.items():
    if not Path(info['path']).exists():
        print(f"⚠️  {info['path']} not found — skipping")
        continue

    print(f"\n{'='*60}")
    print(f"Extracting & Indexing: {info['name']}")
    print('='*60)

    # Pass the unique collection name (insurance_optima_secure, insurance_care_supreme)
    result = await ExtractionPipeline(
        pdf_path        = info['path'],
        chunk_size      = CHUNK_SIZE,
        chunk_overlap   = OVERLAP,
        collection_name = f"insurance_{key}",
        device          = "cuda" 
    ).a_run()

    print(f"Stats: {result.stats}")

## 4 · Initialize the Centralized Engine
With the database populated, we load the embeddings, connection to ChromaDB, reranker, and generator all at once.

In [ ]:
from engine import InsuranceRAGEngine

print("Initializing Centralized RAG Engine... (This may take a moment to load models)")

# Instantiate the engine. It automatically wires up the Retriever, Reranker, and Generator
rag_engine = InsuranceRAGEngine(gemini_api_key=os.environ.get("GEMINI_API_KEY"))

print("✅ Engine Ready!")

## 5 · Single Policy Query Pipeline
Execute a query against a single specified policy document. The engine handles the two-pass adjudicator/explainer logic internally.

In [ ]:
from IPython.display import display, Markdown

query = "Does this policy cover robotic surgery?"
collection = "insurance_care_supreme"

print(f"🔍 Querying {collection} for: '{query}'\n")

# Execute the full pipeline (Retrieve -> Rerank -> Generate)
response_markdown = await rag_engine.a_query_single_policy(
    query=query,
    collection_name=collection,
    history=[],
    retrieve_top_k=20,
    rerank_top_k=4
)

display(Markdown(response_markdown))

## 6 · Cross-Policy Comparison Pipeline
Run a query against both policies simultaneously. The engine retrieves contexts from both databases, compares them independently, and outputs a strategic report.

In [ ]:
query = "Are maternity expenses covered, and what is the exact waiting period for it?"
collection_a = "insurance_optima_secure"
collection_b = "insurance_care_supreme"

print(f"⚖️ Comparing Policies for: '{query}'\n")

comparison_markdown = await rag_engine.a_compare_policies(
    query=query,
    collection_a=collection_a,
    collection_b=collection_b,
    history=[],
    retrieve_top_k=20,
    rerank_top_k=4
)

display(Markdown(comparison_markdown))

## 7 · Diagnostics (Under the Hood)
If the LLM is hallucinating or missing facts, tap into the engine's internal components (`rag_engine.retriever` and `rag_engine.reranker`) to debug the data pipeline directly.

In [ ]:
async def diagnose_retrieval(query: str, collection_name: str, retrieve_top_k: int = 15, rerank_top_k: int = 5):
    """
    Bypasses generation to show exactly what chunks the Cross-Encoder 
    is prioritizing for the LLM.
    """
    print(f"🛠️ DIAGNOSTICS: '{query}'\n")
    
    # 1. Translate Query (Required for Dual-Track Hybrid Search)
    structured_query = await rag_engine.rewriter_chain.ainvoke({"query": query})
    bm25_string = f"{query} {' '.join(structured_query.medical_terms)}".strip()
    vector_string = f"{structured_query.canonical_query} {' '.join(structured_query.expanded_terms)} {' '.join(structured_query.exclusion_terms)} {' '.join(structured_query.policy_sections)}".strip()
    
    # 2. Fetch raw candidates via Asymmetric Ensemble
    search_engine = rag_engine._get_search_engine(collection_name, retrieve_top_k)
    raw_docs = await search_engine.a_search(query, bm25_string, vector_string)
    print(f"Found {len(raw_docs)} initial candidates via Hybrid Fusion.")
    
    # 3. Rerank them
    combined_rerank_string = f"{bm25_string} {vector_string}"
    reranked_docs = await rag_engine.reranker.a_rerank(
        query=combined_rerank_string, 
        documents=raw_docs, 
        top_k=rerank_top_k, 
        return_scores=True
    )
    
    print(f"\n🏆 Top {rerank_top_k} Chunks Selected by Reranker:")
    print("="*70)
    
    for rank, (doc, score) in enumerate(reranked_docs, 1):
        score_str = f"{score:.3f}" if score is not None else "N/A"
        section = doc.metadata.get('section', 'Unknown Section')[:50]
        pages = f"p.{doc.metadata.get('page_start')}–{doc.metadata.get('page_end')}"
        
        print(f"[{rank}] Score: {score_str:<6} | {pages} | {section}")
        print(f"    Preview: {doc.page_content[:150].replace('\\n', ' ')}...\n")

# Run diagnostic check (Ensure top-level await is used)
await diagnose_retrieval("What happens to my cumulative bonus if I make a claim?", "insurance_care_supreme")